In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
## designed to fill large Futures orders efficiently in the market
## Minimise cost
## minimise market impact
## achieve the best possible price
## Futures are standardized derivate contract that obligates the buyer to buy and seller to sell the underlying asset at a predetermined price
## at a specified date in the future
## Two key challenges: (1) buying and selling larger orders pushes price up and down massively (2) if algo takes too long the market price might
## in unfavorable direction


In [4]:
### understand more on the FuTures Algo execution methods
### learn what is Reinforcement learning
### Learn different methods of Reinforcement learning
### try and use diffrent RL library to start with and do no tbuild the model from the start
### need a lightweight code skeleton to get started
##

In [5]:
### the agent needs to make decision how much to trade given market state
## 

In [6]:
## learn more on the different types of reinforcement learning tommorrow and find a base model to build on
## think about where can i get the market data from
## also thinking about how to model the market impact

In [37]:
##  State (what the agent sees)
state = {
    'remaining_quantity': 750,      # contracts left to sell
    'time_elapsed': 0.25,           # 15 minutes into 1-hour execution
    'current_price': 4500.25,       # current futures price
    'bid_ask_spread': 0.25,         # tight spread = good liquidity
    'market_volatility': 'low'      # calm market conditions
}

## Action (what the agent can do)
actions = {
    'AGGRESSIVE': "Market order - sell now at any price",
    'MODERATE':   "Limit order near current bid",
    'PASSIVE':    "Limit order deeper in the book", 
    'WAIT':       "Do nothing for now"
}

# Reward (what we optimize for)
reward = - (Implementation Shortfall)
# Where Implementation Shortfall = (Better Price - Actual Price) × Quantity

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2537105625.py, line 19)

In [38]:
## EPISODE 1:
Start: Need to sell 1000 contracts in 1 hour
→ Agent tries AGGRESSIVE: sells 200 contracts quickly
→ Price drops significantly due to market impact  
→ Reward: -$15,000 (poor execution)

EPISODE 2:
→ Agent tries PASSIVE: waits too long  
→ Market moves against them, misses good prices
→ Reward: -$12,000 (still poor)

EPISODE 100:
→ Agent learns: be MODERATE in calm markets, 
  AGGRESSIVE when volatility spikes
→ Sells small chunks steadily, avoids big moves
→ Reward: -$2,500 (much better!)

SyntaxError: invalid character '→' (U+2192) (2253975625.py, line 3)

In [39]:
def smart_execution_policy(state):
    if state['remaining_quantity'] > 800 and state['time_elapsed'] < 0.1:
        return 'WAIT'        # Don't rush at the beginning
    elif state['market_volatility'] == 'high':
        return 'MODERATE'    # Be careful in volatile markets
    elif state['bid_ask_spread'] > 1.0:
        return 'PASSIVE'     # Wide spreads = be patient
    else:
        return 'MODERATE'    # Steady execution in normal conditions

In [40]:


8:00:00 AM - ORDER RECEIVED
- Sell 1000 ES contracts over 1 hour
- Arrival price: 4500.50

8:00:01 AM - AGENT DECIDES
State: 1000 remaining, 0% time elapsed, low volatility
→ Action: WAIT (don't show your hand immediately)

8:05:00 AM - AGENT DECIDES  
State: 1000 remaining, 8% time elapsed, normal market
→ Action: MODERATE (sell 40 contracts via limit orders)

8:30:00 AM - HALFWAY POINT
State: 520 remaining, 50% time elapsed, volatility increasing
→ Action: MODERATE (stick to the plan)

8:55:00 AM - FINAL PUSH
State: 80 remaining, 92% time elapsed  
→ Action: AGGRESSIVE (use market orders to finish)

9:00:00 AM - ORDER COMPLETE
- All 1000 contracts sold
- Average price: 4500.25
- Implementation Shortfall: 0.25 points × 1000 = $1,250
- Much better than naive approach ($5,000+ shortfall)

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (181226279.py, line 5)

In [41]:
##### copy paste from DeepSeek


import numpy as np
import pandas as pd

## deque is a shortcut function that helps with adding new elements to a given list while deleting the front ones, i.e old ones
from collections import deque

## helps with generating random numbers
import random

## 
from typing import Dict, Tuple, List

## helps with creating Reinforcement learning environments
import gymnasium as gym
from gym import spaces



class FuturesExecutionEnv(gym.Env):
    """Custom Environment for Futures Execution"""
    
    ## We are setting up the room i.e. the constructor
    ## initialising order parameters
    def __init__(self, data: pd.DataFrame, order_size: int = 1000, time_horizon: int = 60): ## int here is a type hint
        
        ## inheriting core functionality/structure from gym.Env
        super(FuturesExecutionEnv, self).__init__()
        
        
        self.data = data  # OHLCV + order book data
        self.order_size = order_size
        self.time_horizon = time_horizon  # in minutes
        self.current_step = 0 ## setting up the time-counter to the initial historical point
        
        # Action space: we require 3 distinctive actions for now: 0=Passive, 1=Moderate, 2=Aggressive
        self.action_space = spaces.Discrete(3)
        
        # State space: we have 8 normalized features and we want to make sure these are normalized
        self.observation_space = spaces.Box(
            low=0, high=1, 
            shape=(8,),  # 8 state features
            dtype=np.float32
        )
        
        self.reset() ## resetting everytime
    
    
    
    ## this part is the AI agent's state or dahsboard
    ## it is compiling 8 pieces of normalized information, i.e  between 0 and 1
    def _get_state(self) -> np.ndarray: ## this function should return a numpy array (vector)
        """Get current state representation"""
        if self.current_step >= len(self.data) - 1: ## if the index of the current_step reaches the end, then it reverts back to the second last item
            self.current_step = len(self.data) - 2
            
        current_data = self.data.iloc[self.current_step]
        next_data = self.data.iloc[self.current_step + 1]
        
        # Normalized state features
        state = np.array([
            self.remaining_quantity / self.order_size,  # % remaining
            self.current_step / len(self.data),         # % time elapsed
            (current_data['spread'] - 0.1) / 2.0,      # normalized spread
            current_data['volatility'] / 0.02,         # normalized volatility
            current_data['imbalance'],                  # order book imbalance (-1 to 1)
            current_data['volume_ratio'],               # volume ratio
            self._get_urgency(),                        # execution urgency, we will define these functions in a bit
            self._get_performance()                     # current performance , we will define these funcitons in a bit 
        ], dtype=np.float32)
        
        return np.clip(state, 0, 1)                     # this limits the values in the state array between 0 and 1
    
    
    ## this funciton output used as an attribute in the state
    ## early on, we want to focus more on the quantity, later on time to be more important as we will run out of trading time
    def _get_urgency(self) -> float:
        """Calculate execution urgency based on remaining time/quantity"""
        time_urgency = (self.current_step / len(self.data)) ## gets close to 1 during the end
        quantity_urgency = 1 - (self.remaining_quantity / self.order_size) ## close to 1 during the end
        
        
        time_weight = time_urgency           ## gets close to 1 later and makes it more urgent
        quantity_weight = 1 - time_urgency   ##  gets it close to 1 in the beginning and reduces to 0 as time becomes more important later on
        
        urgency = quantity_weight * quantity_urgency + time_weight * time_urgency
        
        return flaot(np.clip(urgency, 0 , 1))
    
    
    
    def _get_performance(self) -> float:
        """Calculate current execution performance"""
        if self.quantity_executed == 0:
            return 0.5  # neutral
        
        current_vwap = self.total_value / self.quantity_executed
        arrival_performance = (self.arrival_price - current_vwap) / self.arrival_price
        return np.clip(arrival_performance * 10 + 0.5, 0, 1)
    
    def _calculate_market_impact(self, action: int, quantity: int) -> float:
        """Calculate price impact based on action and quantity"""
        base_impact = {
            0: 0.001,  # Passive: low impact
            1: 0.003,  # Moderate: medium impact  
            2: 0.008   # Aggressive: high impact
        }[action]
        
        size_factor = (quantity / 100) * 0.002
        return base_impact + size_factor
    
    def _get_execution_price(self, action: int, current_price: float) -> float:
        """Get execution price based on action type"""
        spread = self.data.iloc[self.current_step]['spread']
        
        if action == 0:  # Passive - limit order at favorable price
            return current_price - spread * 0.3
        elif action == 1:  # Moderate - limit order near mid
            return current_price - spread * 0.1
        else:  # Aggressive - market order
            return current_price + spread * 0.5  # Pay the spread
    
    def step(self, action: int) -> Tuple[np.ndarray, float, bool, dict]:
        """Execute one time step"""
        current_data = self.data.iloc[self.current_step]
        current_price = current_data['close']
        
        # Determine quantity to execute based on action
        base_quantity = max(1, int(self.order_size * 0.02))  # 2% of total order
        quantity_multiplier = {0: 0.5, 1: 1.0, 2: 2.0}
        quantity = min(
            int(base_quantity * quantity_multiplier[action]),
            self.remaining_quantity
        )
        
        if quantity > 0:
            # Get execution price with market impact
            exec_price = self._get_execution_price(action, current_price)
            impact = self._calculate_market_impact(action, quantity)
            final_price = exec_price * (1 - impact)
            
            # Update execution state
            self.quantity_executed += quantity
            self.remaining_quantity -= quantity
            self.total_value += final_price * quantity
            self.execution_prices.append(final_price)
        
        # Calculate reward
        reward = self._calculate_reward(action, quantity, current_price)
        
        # Move to next time step
        self.current_step += 1
        done = (self.remaining_quantity <= 0 or 
                self.current_step >= len(self.data) - 1 or
                self.current_step >= self.time_horizon)
        
        # Get next state
        next_state = self._get_state()
        
        info = {
            'quantity_executed': self.quantity_executed,
            'remaining_quantity': self.remaining_quantity,
            'avg_price': self.total_value / self.quantity_executed if self.quantity_executed > 0 else 0,
            'step': self.current_step
        }
        
        return next_state, reward, done, info
    
    def _calculate_reward(self, action: int, quantity: int, current_price: float) -> float:
        """Calculate reward for the action taken"""
        if quantity == 0:
            return -0.1  # Small penalty for no execution
        
        # Base reward from execution quality
        if self.quantity_executed > 0:
            current_vwap = self.total_value / self.quantity_executed
            price_improvement = (self.arrival_price - current_vwap) / self.arrival_price
            price_reward = price_improvement * 1000  # Scale up
        else:
            price_reward = 0
        
        # Penalty for market impact
        impact_penalty = -self._calculate_market_impact(action, quantity) * 100
        
        # Urgency bonus/penalty
        urgency = self._get_urgency()
        if self.remaining_quantity > 0:
            time_penalty = -0.01 if urgency > 0.8 else 0
        else:
            time_penalty = 0.1  # Bonus for completing early
        
        # Risk penalty for large remaining quantity late in execution
        risk_penalty = - (self.remaining_quantity / self.order_size) * urgency * 0.1
        
        total_reward = price_reward + impact_penalty + time_penalty + risk_penalty
        return float(total_reward)
    
    def reset(self) -> np.ndarray:
        """Reset the environment"""
        self.current_step = 0
        self.remaining_quantity = self.order_size
        self.quantity_executed = 0
        self.total_value = 0.0
        self.execution_prices = []
        self.arrival_price = self.data.iloc[0]['close']
        
        return self._get_state()

    def __init__(self, state_size: int, action_size: int):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=2000)
        self.gamma = 0.95  # discount rate
        self.epsilon = 1.0  # exploration rate
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.learning_rate = 0.001
        self.model = self._build_model()
        self.target_model = self._build_model()
        self.update_target_network()
    
    def _build_model(self):
        """Build neural network model"""
        from tensorflow.keras.models import Sequential
        from tensorflow.keras.layers import Dense
        from tensorflow.keras.optimizers import Adam
        
        model = Sequential()
        model.add(Dense(24, input_dim=self.state_size, activation='relu'))
        model.add(Dense(24, activation='relu'))
        model.add(Dense(self.action_size, activation='linear'))
        model.compile(loss='mse', optimizer=Adam(learning_rate=self.learning_rate))
        return model
    
    def update_target_network(self):
        """Update target network weights"""
        self.target_model.set_weights(self.model.get_weights())
    
    def remember(self, state, action, reward, next_state, done):
        """Store experience in replay memory"""
        self.memory.append((state, action, reward, next_state, done))
    
    def act(self, state: np.ndarray) -> int:
        """Choose action using epsilon-greedy policy"""
        if np.random.random() <= self.epsilon:
            return random.randrange(self.action_size)
        
        state = state.reshape(1, -1)
        act_values = self.model.predict(state, verbose=0)
        return np.argmax(act_values[0])
    
    def replay(self, batch_size: int = 32):
        """Train on batch from replay memory"""
        if len(self.memory) < batch_size:
            return
        
        minibatch = random.sample(self.memory, batch_size)
        
        for state, action, reward, next_state, done in minibatch:
            state = state.reshape(1, -1)
            next_state = next_state.reshape(1, -1)
            
            target = self.model.predict(state, verbose=0)
            if done:
                target[0][action] = reward
            else:
                t = self.target_model.predict(next_state, verbose=0)
                target[0][action] = reward + self.gamma * np.amax(t[0])
            
            self.model.fit(state, target, epochs=1, verbose=0)
        
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
    
    def load(self, name):
        self.model.load_weights(name)
    
    def save(self, name):
        self.model.save_weights(name)


class FuturesExecutionAlgo:
    """Main Futures Execution Algorithm"""
    
    def __init__(self, symbol: str = "ES"):
        self.symbol = symbol
        self.agent = None
        self.env = None
        self.is_trained = False
    
    def prepare_data(self, raw_data: pd.DataFrame) -> pd.DataFrame:
        """Prepare and feature engineer market data"""
        data = raw_data.copy()
        
        # Calculate features
        data['returns'] = data['close'].pct_change()
        data['volatility'] = data['returns'].rolling(20).std().fillna(0.01)
        data['spread'] = (data['ask'] - data['bid']).fillna(0.25)
        data['imbalance'] = ((data['bid_volume'] - data['ask_volume']) / 
                           (data['bid_volume'] + data['ask_volume'])).fillna(0)
        data['volume_ratio'] = (data['volume'] / data['volume'].rolling(50).mean()).fillna(1)
        
        # Normalize features
        data['volatility'] = data['volatility'].clip(0, 0.02)
        data['spread'] = data['spread'].clip(0.1, 2.0)
        
        return data.dropna()
    
    def train(self, data: pd.DataFrame, episodes: int = 1000):
        """Train the RL agent"""
        print("Starting training...")
        
        # Prepare environment
        processed_data = self.prepare_data(data)
        self.env = FuturesExecutionEnv(processed_data)
        
        # Initialize agent
        state_size = self.env.observation_space.shape[0]
        action_size = self.env.action_space.n
        self.agent = DQNAgent(state_size, action_size)
        
        # Training metrics
        scores = []
        avg_scores = []
        
        for episode in range(episodes):
            state = self.env.reset()
            state = state.reshape(1, -1)
            total_reward = 0
            done = False
            
            while not done:
                action = self.agent.act(state)
                next_state, reward, done, info = self.env.step(action)
                next_state = next_state.reshape(1, -1)
                
                self.agent.remember(state, action, reward, next_state, done)
                state = next_state
                total_reward += reward
            
            scores.append(total_reward)
            avg_score = np.mean(scores[-100:])
            avg_scores.append(avg_score)
            
            if episode % 100 == 0:
                print(f"Episode {episode}, Score: {total_reward:.2f}, Avg Score: {avg_score:.2f}, Epsilon: {self.agent.epsilon:.3f}")
            
            self.agent.replay()
            
            if episode % 50 == 0:
                self.agent.update_target_network()
        
        self.is_trained = True
        print("Training completed!")
        return scores, avg_scores
    
    def execute_order(self, live_data: pd.DataFrame, order_size: int) -> Dict:
        """Execute a live order using trained agent"""
        if not self.is_trained:
            raise ValueError("Agent must be trained before execution")
        
        processed_data = self.prepare_data(live_data)
        self.env = FuturesExecutionEnv(processed_data, order_size=order_size)
        
        state = self.env.reset()
        done = False
        execution_log = []
        
        while not done:
            action = self.agent.act(state)
            next_state, reward, done, info = self.env.step(action)
            
            execution_log.append({
                'step': info['step'],
                'action': action,
                'quantity_executed': info['quantity_executed'],
                'remaining_quantity': info['remaining_quantity'],
                'average_price': info['avg_price'],
                'reward': reward
            })
            
            state = next_state
        
        # Execution summary
        summary = {
            'total_quantity': order_size,
            'executed_quantity': info['quantity_executed'],
            'average_execution_price': info['avg_price'],
            'arrival_price': self.env.arrival_price,
            'implementation_shortfall': (self.env.arrival_price - info['avg_price']) * order_size,
            'completion_rate': info['quantity_executed'] / order_size,
            'execution_log': execution_log
        }
        
        return summary


# Example usage
if __name__ == "__main__":
    # Generate sample data (in practice, use real market data)
    def generate_sample_data(num_points: int = 1000) -> pd.DataFrame:
        dates = pd.date_range('2024-01-01', periods=num_points, freq='1min')
        data = pd.DataFrame({
            'date': dates,
            'open': 4500 + np.cumsum(np.random.randn(num_points) * 0.1),
            'high': 0,
            'low': 0, 
            'close': 0,
            'volume': np.random.randint(1000, 10000, num_points),
            'bid': 0,
            'ask': 0,
            'bid_volume': 0,
            'ask_volume': 0
        })
        
        data['close'] = data['open'] + np.random.randn(num_points) * 0.05
        data['high'] = data[['open', 'close']].max(axis=1) + np.random.rand() * 0.1
        data['low'] = data[['open', 'close']].min(axis=1) - np.random.rand() * 0.1
        data['bid'] = data['close'] - 0.25
        data['ask'] = data['close'] + 0.25
        data['bid_volume'] = np.random.randint(100, 1000, num_points)
        data['ask_volume'] = np.random.randint(100, 1000, num_points)
        
        return data.set_index('date')
    
    # Initialize and train the algo
    algo = FuturesExecutionAlgo("ES")
    
    # Generate training data
    training_data = generate_sample_data(5000)
    
    # Train the model
    scores, avg_scores = algo.train(training_data, episodes=500)
    
    # Execute a sample order
    live_data = generate_sample_data(100)
    result = algo.execute_order(live_data, order_size=500)
    
    print("\nExecution Summary:")
    print(f"Executed: {result['executed_quantity']}/{result['total_quantity']}")
    print(f"Average Price: {result['average_execution_price']:.2f}")
    print(f"Arrival Price: {result['arrival_price']:.2f}")
    print(f"Implementation Shortfall: ${result['implementation_shortfall']:.2f}")

ModuleNotFoundError: No module named 'gymnasium'